In [1]:
"""Metrics and patient-level bootstrap confidence intervals.

This module provides:
    - False-negative rate (FNR)
    - Female-minus-male FNR gap
    - Patient-level bootstrap confidence intervals

Bootstrap resampling is clustered by Patient ID so that all images
belonging to a sampled patient are retained together.

The default bootstrap uses 1,000 patient-level resamples and a
two-sided 95% percentile confidence interval.
"""

from __future__ import annotations

from typing import Callable, Dict, Optional, Sequence, Tuple

import numpy as np
import pandas as pd


MetricFunction = Callable[[pd.DataFrame], float]


def false_negative_rate(
    y_true: Sequence[int],
    y_pred: Sequence[int],
) -> float:
    """Calculate false-negative rate.

    FNR = false negatives / all positive ground-truth cases.
    """

    y_true_array = np.asarray(y_true)
    y_pred_array = np.asarray(y_pred)

    if len(y_true_array) != len(y_pred_array):
        raise ValueError(
            "y_true and y_pred must have the same number of observations."
        )

    if len(y_true_array) == 0:
        raise ValueError("Cannot calculate FNR from empty data.")

    positive_cases = y_true_array == 1

    if not np.any(positive_cases):
        raise ValueError(
            "FNR is undefined because there are no positive ground-truth cases."
        )

    false_negatives = np.sum(
        positive_cases & (y_pred_array == 0)
    )

    return float(false_negatives / np.sum(positive_cases))


def fnr_by_sex(
    df: pd.DataFrame,
    label_column: str,
    prediction_column: str,
    sex_column: str = "Patient Sex",
) -> Dict[str, float]:
    """Calculate FNR separately for female and male patients."""

    required_columns = {
        label_column,
        prediction_column,
        sex_column,
    }

    missing = required_columns.difference(df.columns)

    if missing:
        raise KeyError(
            f"Missing required columns: {sorted(missing)}"
        )

    female = df[df[sex_column] == "F"]
    male = df[df[sex_column] == "M"]

    if female.empty:
        raise ValueError("No female observations found.")

    if male.empty:
        raise ValueError("No male observations found.")

    return {
        "fnr_female": false_negative_rate(
            female[label_column],
            female[prediction_column],
        ),
        "fnr_male": false_negative_rate(
            male[label_column],
            male[prediction_column],
        ),
    }


def fnr_sex_gap(
    df: pd.DataFrame,
    label_column: str,
    prediction_column: str,
    sex_column: str = "Patient Sex",
) -> float:
    """Calculate the sex gap:

        S = FNR(female) - FNR(male)
    """

    fnrs = fnr_by_sex(
        df=df,
        label_column=label_column,
        prediction_column=prediction_column,
        sex_column=sex_column,
    )

    return fnrs["fnr_female"] - fnrs["fnr_male"]


def patient_bootstrap(
    df: pd.DataFrame,
    metric: MetricFunction,
    patient_id_column: str = "Patient ID",
    n_resamples: int = 1000,
    confidence_level: float = 0.95,
    random_seed: Optional[int] = None,
) -> Tuple[float, float, float]:
    """Patient-level bootstrap for any metric.

    Each bootstrap replicate samples Patient IDs with replacement.
    Every image belonging to a sampled patient is included together.

    Returns:
        point_estimate
        lower_confidence_bound
        upper_confidence_bound
    """

    if patient_id_column not in df.columns:
        raise KeyError(
            f"Data must contain patient ID column "
            f"{patient_id_column!r}."
        )

    if df.empty:
        raise ValueError("Cannot bootstrap an empty dataset.")

    if n_resamples <= 0:
        raise ValueError("n_resamples must be positive.")

    if not 0 < confidence_level < 1:
        raise ValueError(
            "confidence_level must be between 0 and 1."
        )

    patient_ids = df[patient_id_column].dropna().unique()

    if len(patient_ids) == 0:
        raise ValueError("No valid patient IDs found.")

    point_estimate = float(metric(df))

    rng = np.random.default_rng(random_seed)

    bootstrap_estimates = []

    for _ in range(n_resamples):
        sampled_patient_ids = rng.choice(
            patient_ids,
            size=len(patient_ids),
            replace=True,
        )

        bootstrap_parts = [
            df[df[patient_id_column] == patient_id]
            for patient_id in sampled_patient_ids
        ]

        bootstrap_df = pd.concat(
            bootstrap_parts,
            ignore_index=True,
        )

        try:
            estimate = float(metric(bootstrap_df))
        except ValueError:
            # The bootstrap sample can occasionally lack a required
            # subgroup or outcome class. Skip that replicate.
            continue

        if np.isfinite(estimate):
            bootstrap_estimates.append(estimate)

    if not bootstrap_estimates:
        raise RuntimeError(
            "No valid bootstrap replicates were produced."
        )

    bootstrap_estimates = np.asarray(
        bootstrap_estimates,
        dtype=float,
    )

    alpha = 1.0 - confidence_level

    lower = float(
        np.percentile(
            bootstrap_estimates,
            100 * (alpha / 2),
        )
    )

    upper = float(
        np.percentile(
            bootstrap_estimates,
            100 * (1 - alpha / 2),
        )
    )

    return point_estimate, lower, upper


def bootstrap_fnr_sex_gap(
    df: pd.DataFrame,
    label_column: str,
    prediction_column: str,
    sex_column: str = "Patient Sex",
    patient_id_column: str = "Patient ID",
    n_resamples: int = 1000,
    confidence_level: float = 0.95,
    random_seed: Optional[int] = None,
) -> Dict[str, float]:
    """Calculate the female-minus-male FNR gap and its 95% CI."""

    metric = lambda data: fnr_sex_gap(
        df=data,
        label_column=label_column,
        prediction_column=prediction_column,
        sex_column=sex_column,
    )

    estimate, lower, upper = patient_bootstrap(
        df=df,
        metric=metric,
        patient_id_column=patient_id_column,
        n_resamples=n_resamples,
        confidence_level=confidence_level,
        random_seed=random_seed,
    )

    return {
        "metric": "FNR sex gap",
        "estimate": estimate,
        "ci_lower": lower,
        "ci_upper": upper,
        "confidence_level": confidence_level,
        "n_resamples": n_resamples,
    }


def bootstrap_metric(
    df: pd.DataFrame,
    metric: MetricFunction,
    patient_id_column: str = "Patient ID",
    n_resamples: int = 1000,
    confidence_level: float = 0.95,
    random_seed: Optional[int] = None,
) -> Dict[str, float]:
    """Return a point estimate and patient-level bootstrap CI for any metric."""

    estimate, lower, upper = patient_bootstrap(
        df=df,
        metric=metric,
        patient_id_column=patient_id_column,
        n_resamples=n_resamples,
        confidence_level=confidence_level,
        random_seed=random_seed,
    )

    return {
        "estimate": estimate,
        "ci_lower": lower,
        "ci_upper": upper,
        "confidence_level": confidence_level,
        "n_resamples": n_resamples,
    }